In [ ]:
pip install tensorflow numpy matplotlib pillow notebook jupyter pandas

In [ ]:
import sys
print(sys.version)

import tensorflow as tf
print(tf.__version__)

In [ ]:
import tensorflow as tf
from tensorflow.keras import models, layers
import matplotlib.pyplot as plt

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = 256
CHANNELS = 3
EPOCHS = 50

In [ ]:
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "PlantVillage",
    shuffle=True,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

In [ ]:
class_names=dataset.class_names
class_names

In [ ]:
plt.figure(figsize=(10, 10))

for image_batch, labels_batch in dataset.take(1):
    print(image_batch.shape)
    print(labels_batch.numpy())

    for i in range(12):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(image_batch[i].numpy().astype("uint8"))
        plt.title(class_names[labels_batch[i]])
        plt.axis("off")

In [ ]:
train_size=0.8
len(dataset)*train_size

In [ ]:
test_ds = dataset.skip(54)
len(test_ds)

In [ ]:
val_size=0.1
len(dataset)*val_size

In [ ]:
val_ds=test_ds.take(6)
len(val_ds)

In [ ]:
train_size = 0.8
val_size = 0.1

train_ds = dataset.take(54)

remaining_ds = dataset.skip(54)

val_ds = remaining_ds.take(6)

test_ds = remaining_ds.skip(6)

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))

In [ ]:
def get_dataset_partitions_tf(ds, train_split=0.8, val_split=0.1, test_split=0.1, shuffle=True, shuffle_size=10000):
    assert (train_split + val_split + test_split) == 1

    ds_size = len(ds)

    if shuffle:
        ds = ds.shuffle(shuffle_size, seed=12)

    train_size = int(train_split * ds_size)
    val_size = int(val_split * ds_size)

    train_ds = ds.take(train_size)
    val_ds = ds.skip(train_size).take(val_size)
    test_ds = ds.skip(train_size).skip(val_size)

    return train_ds, val_ds, test_ds

In [ ]:
train_ds,val_ds,test_ds=get_dataset_partitions_tf(dataset)

In [ ]:
len(train_ds)

In [ ]:
len(val_ds)

In [ ]:
len(test_ds)

In [ ]:
train_ds= train_ds.cache().shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
val_ds= train_ds.cache().shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
test_ds= test_ds.cache().shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)

In [ ]:
for image_batch, labels_batch in dataset.take(1):
    print(image_batch[0].numpy()/255)

In [ ]:
resize_and_rescale = tf.keras.Sequential([
    layers.Resizing(IMAGE_SIZE, IMAGE_SIZE),
    layers.Rescaling(1.0 / 255)
])

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
])

In [ ]:
input_shape = (IMAGE_SIZE, IMAGE_SIZE, CHANNELS)
n_classes = 3

model = models.Sequential([
    layers.Input(shape=input_shape),

    resize_and_rescale,
    data_augmentation,

    layers.Conv2D(32, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(n_classes, activation='softmax'),
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_ds,
    batch_size=BATCH_SIZE,
    validation_data=val_ds,
    verbose=1,
    epochs=EPOCHS
)

In [ ]:
len(test_ds)

In [ ]:
scores = model.evaluate(test_ds)

In [ ]:
scores

In [ ]:
history.params #history.history.keys()

In [ ]:
model.save("potato_disease_model.keras")

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
from tkinter import Tk
from tkinter.filedialog import askopenfilename

# Load model
model = load_model("potato_disease_model.keras")

# Hide main tkinter window
Tk().withdraw()

# Open file picker
img_path = askopenfilename()

# Load image
img = image.load_img(img_path, target_size=(256, 256))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

# Prediction
prediction = model.predict(img_array)

classes = ["Early Blight", "Late Blight", "Healthy"]
result = classes[np.argmax(prediction)]

print("Prediction:", result)

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np

# Load model
model = load_model("potato_disease_model.keras")

# image path (tumhara file ngame)
img_path = "potato.jpeg"

# Load and preprocess image
img = image.load_img(img_path, target_size=(256, 256))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

# Prediction
prediction = model.predict(img_array)

classes = ["Early Blight", "Late Blight", "Healthy"]
result = classes[np.argmax(prediction)]

print("Prediction:", result)

In [ ]:
from tensorflow.keras.models import load_model
print("loading...")
model = load_model("potato_disease_model.keras")
print("Done!")

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img_path = "potato.jpeg"  # ← ye naam hai tumhari image ka

img = image.load_img(img_path, target_size=(256, 256))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
classes = ["Early Blight", "Late Blight", "Healthy"]
print("Prediction:", classes[np.argmax(prediction)])

In [ ]:
test_images = [
    "Earlyblight1.jpeg",
    "Earlyblight2.jpeg",
    "Earlyblight3.jpeg",
    "Healthyleaf1.jpeg",
    "Healthyleaf2.jpeg",
    "Healthyleaf3.jpeg",
    "LateBlight1.jpeg",
    "LateBlight2.jpeg",
    "LateBlight3.jpeg",
]

classes = ["Early Blight", "Late Blight", "Healthy"]

for img_name in test_images:
    img = image.load_img(img_name, target_size=(256, 256))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    result = classes[np.argmax(prediction)]
    print(f"{img_name} → {result}")

In [ ]:
import tensorflow as tf
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "PlantVillage",
    shuffle=False,
    image_size=(256, 256),
    batch_size=32
)
print(dataset.class_names)

In [ ]:
import tensorflow as tf
from tensorflow.keras import models, layers
import numpy as np
from tensorflow.keras.preprocessing import image

# Settings
IMAGE_SIZE = 256
BATCH_SIZE = 32
EPOCHS = 20

# Dataset load
dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "PlantVillage",
    shuffle=True,
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

print("Classes:", dataset.class_names)

# Split
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
train_ds = dataset.take(train_size)
val_ds = dataset.skip(train_size).take(val_size)
test_ds = dataset.skip(train_size + val_size)

# Model
model = models.Sequential([
    layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
    layers.Rescaling(1.0/255),
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# Train
print("Training shuru...")
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=1)

# Save
model.save("potato_disease_model.keras")
print("Model saved!")

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np

model = load_model("potato_disease_model.keras")
print("Model ready!")

In [ ]:
test_images = [
    "Earlyblight1.jpeg",
    "Earlyblight2.jpeg",
    "Earlyblight3.jpeg",
    "Healthyleaf1.jpeg",
    "Healthyleaf2.jpeg",
    "Healthyleaf3.jpeg",
    "LateBlight1.jpeg",
    "LateBlight2.jpeg",
    "LateBlight3.jpeg",
]

classes = ["Early Blight", "Late Blight", "Healthy"]

for img_name in test_images:
    img = image.load_img(img_name, target_size=(256, 256))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_array, verbose=0)
    result = classes[np.argmax(prediction)]
    print(f"{img_name} → {result}")

!pip install streamlit

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
import numpy as np
import io
from PIL import Image

model = load_model("potato_disease_model.keras")
classes = ["Early Blight", "Late Blight", "Healthy"]

upload = widgets.FileUpload(accept='image/*', multiple=False)
button = widgets.Button(description="🔍 Predict!", button_style='success')
output = widgets.Output()
title = widgets.HTML("<h2>🥔 Potato Disease Detector</h2>")
instruction = widgets.HTML("<p>Image upload karo aur Predict button dabao!</p>")

def on_predict(b):
    with output:
        clear_output()
        if upload.value:
            img_data = upload.value[0]['content']
            img = Image.open(io.BytesIO(img_data)).resize((256, 256))
            img_array = np.array(img)
            img_array = np.expand_dims(img_array, axis=0)
            prediction = model.predict(img_array, verbose=0)
            result = classes[np.argmax(prediction)]
            confidence = round(float(np.max(prediction)) * 100, 2)
            if result == "Healthy":
                color = "green"
                emoji = "✅"
            else:
                color = "red"
                emoji = "⚠️"
            display(widgets.HTML(f"""
                <div style='padding:15px; background:#f0f0f0; border-radius:10px;'>
                    <h3 style='color:{color}'>{emoji} Result: {result}</h3>
                    <p>Confidence: <b>{confidence}%</b></p>
                </div>
            """))
        else:
            display(widgets.HTML("<p style='color:red'>⚠️ Pehle image upload karo!</p>"))

button.on_click(on_predict)
display(title, instruction, upload, button, output)

HTML(value='<h2>🥔 Potato Disease Detector</h2>')

HTML(value='<p>Image upload karo aur Predict button dabao!</p>')

FileUpload(value=(), accept='image/*', description='Upload')

Button(button_style='success', description='🔍 Predict!', style=ButtonStyle())

Output()